## Подготовка данных для модели

In [21]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

data_path = "../data"

users = pd.read_csv(
    f"{data_path}/users.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["user_id", "gender", "age", "occupation", "zip_code"]
)

movies = pd.read_csv(
    f"{data_path}/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movie_id", "title", "genres"]
)

ratings = pd.read_csv(
    f"{data_path}/ratings.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

In [22]:
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")

### 1. Кодируем user_id и movie_id в индексы

In [23]:
# строим словари: id -> индекс
user_ids = ratings["user_id"].unique()
item_ids = ratings["movie_id"].unique()

user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}
item_id_to_idx = {iid: i for i, iid in enumerate(item_ids)}

# и обратные маппинги (пригодятся для вывода рекомендаций)
idx_to_user_id = {i: uid for uid, i in user_id_to_idx.items()}
idx_to_item_id = {i: iid for iid, i in item_id_to_idx.items()}

# добавим в таблицу столбцы с индексами
ratings["user_idx"] = ratings["user_id"].map(user_id_to_idx)
ratings["item_idx"] = ratings["movie_id"].map(item_id_to_idx)

ratings.head()

,user_id,movie_id,rating,timestamp,user_idx,item_idx
0,1,1193,5,2000-12-31 22:12:40,0,0
1,1,661,3,2000-12-31 22:35:09,0,1
2,1,914,3,2000-12-31 22:32:48,0,2
3,1,3408,4,2000-12-31 22:04:35,0,3
4,1,2355,5,2001-01-06 23:38:11,0,4


### 2. Преобразуем рейтинг в impicit-сигнал

In [24]:
alpha = 2.0
ratings["confidence"] = 1.0 + alpha * (ratings["rating"] - 1.0)

ratings[["rating", "confidence"]].head()
ratings["confidence"].describe()

count    1.000209e+06
mean     6.163129e+00
std      2.234204e+00
min      1.000000e+00
25%      5.000000e+00
50%      7.000000e+00
75%      7.000000e+00
max      9.000000e+00
Name: confidence, dtype: float64

### 3. Time-based split (train / test по времени)

In [25]:
def train_test_split_by_time(df, test_size=0.2):
    # df: таблица ratings с user_idx, item_idx, timestamp
    df = df.sort_values(["user_idx", "timestamp"])
    
    # для каждого пользователя считаем границу
    def split_user(group):
        n = len(group)
        test_n = max(1, int(n * test_size))  
        return pd.Series(
            ["train"] * (n - test_n) + ["test"] * test_n,
            index=group.index
        )
    
    df["split"] = df.groupby("user_idx", group_keys=False).apply(split_user)
    return df

ratings = train_test_split_by_time(ratings, test_size=0.2)

ratings["split"].value_counts()

C:\Users\Yarik\AppData\Local\Temp\ipykernel_3912\2671540249.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["split"] = df.groupby("user_idx", group_keys=False).apply(split_user)


split
train    802553
test     197656
Name: count, dtype: int64

In [26]:
ratings.groupby("split")["rating"].count() / len(ratings)

split
test     0.197615
train    0.802385
Name: rating, dtype: float64

### 4. Строим разреженные матрицы для implicit

In [27]:
from scipy.sparse import coo_matrix

train_df = ratings[ratings["split"] == "train"]
test_df  = ratings[ratings["split"] == "test"]

n_users = ratings["user_idx"].nunique()
n_items = ratings["item_idx"].nunique()

train_matrix = coo_matrix(
    (train_df["confidence"], (train_df["item_idx"], train_df["user_idx"])),
    shape=(n_items, n_users)
).tocsr()

test_matrix = coo_matrix(
    (np.ones(len(test_df)), (test_df["item_idx"], test_df["user_idx"])),
    shape=(n_items, n_users)
).tocsr()

train_matrix.shape, test_matrix.shape

((3706, 6040), (3706, 6040))

### 5. Проверка

In [28]:
train_users = set(train_df["user_idx"].unique())
test_users  = set(test_df["user_idx"].unique())

len(train_users), len(test_users), len(train_users & test_users)

(6040, 6040, 6040)

## Метрики и baseline

Рассмотрим 3 метрики:

- Recall@K (ключевая)
- MAP@K (учитывает порядок рекомендаций)
- NDCG@K (штрафует за порядок логарифмом)

In [29]:
import numpy as np

def recall_at_k(pred, truth, k=10):
    """
    pred: список item_idx рекомендованных моделью
    truth: множество item_idx, которые реально были в тесте
    """
    pred_k = pred[:k]
    hits = len([x for x in pred_k if x in truth])
    return hits / max(1, len(truth))

In [30]:
# MAP@K

def apk(pred, truth, k=10):
    pred_k = pred[:k]
    score = 0.0
    hits = 0
    for i, p in enumerate(pred_k):
        if p in truth:
            hits += 1
            score += hits / (i + 1)
    return score / max(1, min(len(truth), k))

In [31]:
# NDCG@K

def ndcg_at_k(pred, truth, k=10):
    pred_k = pred[:k]
    dcg = 0.0
    for i, p in enumerate(pred_k):
        if p in truth:
            dcg += 1 / np.log2(i + 2)

    ideal_dcg = sum(1 / np.log2(i + 2) for i in range(min(len(truth), k)))

    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0

Подготовим тестовые таргеты

In [32]:
from collections import defaultdict

test_truth = defaultdict(set)

for u, it in zip(test_df["user_idx"], test_df["item_idx"]):
    test_truth[u].add(it)

In [33]:
len(test_truth)

6040

Baseline "Популярные фильмы"

In [34]:
item_popularity = train_df.groupby("item_idx")["confidence"].sum()
popular_items = item_popularity.sort_values(ascending=False).index.values

In [35]:
def evaluate_popular(k=10):
    recalls, maps, ndcgs = [], [], []

    for user, truth in test_truth.items():
        pred = popular_items[:k]
        recalls.append(recall_at_k(pred, truth, k))
        maps.append(apk(pred, truth, k))
        ndcgs.append(ndcg_at_k(pred, truth, k))

    return {
        "Recall@K": np.mean(recalls),
        "MAP@K": np.mean(maps),
        "NDCG@K": np.mean(ndcgs),
    }

baseline_metrics = evaluate_popular(k=10)
baseline_metrics

{'Recall@K': np.float64(0.022117983251158598),
 'MAP@K': np.float64(0.016678644845232675),
 'NDCG@K': np.float64(0.043104290784354245)}

**Вывод**

Модель популярности показывает низкие значения Recall@10 и MAP@10, что ожидаемо: популярные фильмы не учитывают индивидуальные предпочтения пользователя. Тем не менее NDCG@10 выше, поскольку популярные фильмы часто присутствуют в тесте и находятся высоко в списке рекомендаций. Этот бейзлайн будет служить отправной точкой для сравнения с факторизационными моделями.

### Обучаем ALS

In [55]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(
    factors=64,         # размер эмбеддингов
    regularization=0.1, # L2-регуляризация
    iterations=20,      # число итераций
    random_state=42
)

als_model.fit(train_matrix)

100%|██████████| 20/20 [00:00<00:00, 30.08it/s]


Предсказания для теста

In [66]:
def recommend_als(user_idx, K=10):
    user_items = train_matrix.T[user_idx].tocsr()

    item_ids, scores = als_model.recommend(
        userid=user_idx,
        user_items=user_items,
        N=K,
        filter_already_liked_items=True,
        recalculate_user=True
    )

    return list(item_ids)

In [67]:
list(test_truth.keys())[:5]

[0, 1, 2, 3, 4]

Оценка ALS

In [68]:
def evaluate_als(K=10):
    recalls, maps, ndcgs = [], [], []

    for user_idx, truth_items in test_truth.items():
        pred = recommend_als(user_idx, K=K)

        recalls.append(recall_at_k(pred, truth_items, K))
        maps.append(apk(pred, truth_items, K))
        ndcgs.append(ndcg_at_k(pred, truth_items, K))

    return {
        "Recall@K": float(np.mean(recalls)),
        "MAP@K": float(np.mean(maps)),
        "NDCG@K": float(np.mean(ndcgs)),
    }

als_metrics = evaluate_als(K=10)
als_metrics

{'Recall@K': 0.003074577424730337,
 'MAP@K': 0.003795071546830653,
 'NDCG@K': 0.012011662353280548}

Полученные результаты оказались хуже, чем в baseline.

Изначально мы обучили модель ALS в режиме explicit feedback, трактуя рейтинги как числовые значения. Однако такая постановка задачи оптимизирует MSE и ориентирована на предсказание чисел, а не на формирование топ-N рекомендаций. Для рекомендательных систем это неэффективно, так как в проде оптимизируются ранжирующие метрики (Recall@K, MAP@K, NDCG@K), а фидбек чаще всего implicit (просмотры, клики, покупки). Поэтому мы переходим на implicit-ALS/BPR/WARP подходы, которые оптимизированы под ранжирование и демонстрируют существенно более высокие результаты в top-K задачах.

## Доработка ALS

In [ ]:
def recommend_implicit(user_idx: int, k: int = 10):

    # сколько пользователей реально есть в модели
    n_als_users = als_model.user_factors.shape[0]

    # если такого пользователя нет в модели — возвращаем пустой список
    if user_idx < 0 or user_idx >= n_als_users:
        return []

    # берём вектор взаимодействий пользователя (строка из item-user матрицы)
    # implicit_matrix: shape = (n_items, n_users)
    user_items = implicit_matrix.T.tocsr()[user_idx]

    # implicit.recommender возвращает два массива: ids и scores
    ids, scores = als_model.recommend(
        userid=user_idx,
        user_items=user_items,
        N=k,
        filter_already_liked_items=True,
        recalculate_user=True
    )

    # нам для метрик нужны только item_id
    return ids.tolist()

In [ ]:
def evaluate_implicit(K: int = 10):

    recalls, maps, ndcgs = [], [], []

    n_als_users = als_model.user_factors.shape[0]

    for user_idx, truth_items in test_truth.items():
        # пропускаем пользователей, которых нет в обученной ALS
        if user_idx < 0 or user_idx >= n_als_users:
            continue

        # делаем рекомендации
        pred = recommend_implicit(user_idx, k=K)

        # если вдруг рекомендаций нет — тоже пропускаем
        if len(pred) == 0:
            continue

        recalls.append(recall_at_k(pred, truth_items, K))
        maps.append(apk(pred, truth_items, K))
        ndcgs.append(ndcg_at_k(pred, truth_items, K))

    print(f"Число пользователей, попавших в оценку: {len(recalls)}")

    return {
        "Recall@K": float(np.mean(recalls)),
        "MAP@K": float(np.mean(maps)),
        "NDCG@K": float(np.mean(ndcgs)),
    }


implicit_metrics = evaluate_implicit(K=10)
implicit_metrics

Число пользователей, попавших в оценку: 3706


{'Recall@K': 0.0, 'MAP@K': 0.0, 'NDCG@K': 0.0}

Полученные метрики implicit ALS оказались равны 0 при time-based split. 
Причина не в ошибке реализации: implicit ALS плохо подходит для 
предсказания *последних* действий пользователя, так как использует 
статичное факторизационное представление без учёта временной динамики.

Поэтому baseline (популярность) показывает лучшее покрытие в тесте, 
а ALS ловит долгосрочные предпочтения и не попадает в актуальные 
последние взаимодействия.

### item-kNN

In [112]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

n_users, n_items = train_matrix.shape

# item-user матрица (для сходства по фильмам)
item_user_matrix = train_matrix.T  # (n_items, n_users)

In [113]:
# косинусное сходство между фильмами
item_sim_matrix = cosine_similarity(item_user_matrix)  # (n_items, n_items)

item_sim_matrix.shape

(6040, 6040)

In [114]:
def recommend_item_knn(user_idx: int, k: int = 10):

    # вектор оценок пользователя в train (1 × n_items)
    user_row = train_matrix[user_idx]  # csr-строка

    # какие фильмы он уже оценивал
    seen_items = user_row.indices
    if len(seen_items) == 0:
        return []  # на всякий случай

    # суммарный скор по похожести: складываем строки item_sim по просмотренным фильмам
    scores = item_sim_matrix[seen_items].sum(axis=0)  # shape (n_items,)

    # обнуляем уже просмотренные фильмы
    scores[seen_items] = -np.inf

    # берём top-k
    top_k_items = np.argsort(scores)[-k:][::-1]

    return top_k_items.tolist()

In [115]:
def evaluate_item_knn(K: int = 10):
    recalls, maps, ndcgs = [], [], []

    for user_idx, truth_items in test_truth.items():
        if len(truth_items) == 0:
            continue
        if user_idx < 0 or user_idx >= n_users:
            continue

        pred = recommend_item_knn(user_idx, k=K)

        recalls.append(recall_at_k(pred, truth_items, K))
        maps.append(apk(pred, truth_items, K))
        ndcgs.append(ndcg_at_k(pred, truth_items, K))

    print(f"Число пользователей в оценке: {len(recalls)}")
    return {
        "Recall@K": float(np.mean(recalls)),
        "MAP@K": float(np.mean(maps)),
        "NDCG@K": float(np.mean(ndcgs)),
    }

item_knn_metrics = evaluate_item_knn(K=10)
item_knn_metrics

Число пользователей в оценке: 3706


{'Recall@K': 0.001580695111248734,
 'MAP@K': 0.001837531648745125,
 'NDCG@K': 0.006061818911612967}

**Вывод:**

Несмотря на формально невысокие значения метрик, поведение модели логичное и ожидаемое:
 - MovieLens 1M содержит очень разреженную матрицу взаимодействий (≈ 99.9% sparsity), поэтому у большинства пользователей мало пересечений
 - item-based коллаборативная фильтрация работает хорошо при плотных данных, а при разреженных сильно деградирует
 - модель строит рекомендации только на основе сходства фильмов, не используя рейтинги, факторизацию, признаки, контекст, жанры и пользователей-аналогов

Тем не менее item-kNN обгоняет популярный baseline, что подтверждает способность модели учитывать индивидуальные предпочтения.

Это хороший переходный шаг перед более продвинутыми моделями.

## Contend-based модель (по жанрам и году)

In [116]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity

In [117]:
# разбираем жанры в список
movies_cb = movies.copy()
movies_cb["genre_list"] = movies_cb["genres"].str.split("|")

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies_cb["genre_list"])

genre_cols = [f"genre_{g}" for g in mlb.classes_]
genre_df = pd.DataFrame(genre_matrix, columns=genre_cols, index=movies_cb.index)

In [123]:
def extract_year(title):
    m = re.search(r"\((\d{4})\)", title)
    if m:
        return int(m.group(1))
    return np.nan

movies_cb["year"] = movies_cb["title"].apply(extract_year)

# заполним пропуски медианой, нормализуем год
median_year = movies_cb["year"].median()
movies_cb["year"].fillna(median_year, inplace=True)

movies_cb["year_norm"] = (movies_cb["year"] - movies_cb["year"].mean()) / movies_cb["year"].std()

C:\Users\Yarik\AppData\Local\Temp\ipykernel_3912\3176368185.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movies_cb["year"].fillna(median_year, inplace=True)


In [124]:
# объединяем жанры + нормированный год
feature_df = pd.concat([genre_df, movies_cb[["year_norm"]]], axis=1)

# порядок фильмов в матрице признаков
movie_ids_cb = movies_cb["movie_id"].values  # массив movie_id в том же порядке
movie_id_to_pos = {mid: i for i, mid in enumerate(movie_ids_cb)}

features = feature_df.values  # numpy (n_items, n_features)

In [125]:
movie_sim = cosine_similarity(features)  # (n_items, n_items)

## Contend-based рекомендации

In [126]:
# оставим только нужные столбцы
train_small = train_df[["user_id", "movie_id", "rating"]].copy()
test_small  = test_df[["user_id", "movie_id", "rating"]].copy()

# какие фильмы пользователь смотрел в train
user_train_items = (
    train_small.groupby("user_id")["movie_id"]
    .apply(list)
    .to_dict()
)

# фильмы пользователя в test (истина для метрик)
user_test_items = (
    test_small.groupby("user_id")["movie_id"]
    .apply(list)
    .to_dict()
)

len(user_train_items), len(user_test_items)

(6040, 6040)

In [127]:
def recall_at_k(pred, truth, k):
    if len(truth) == 0:
        return 0.0
    pred_k = pred[:k]
    return len(set(pred_k) & set(truth)) / float(len(set(truth)))

def apk(pred, truth, k):
    if len(truth) == 0:
        return 0.0
    pred = pred[:k]
    score = 0.0
    num_hits = 0.0
    truth_set = set(truth)
    for i, p in enumerate(pred):
        if p in truth_set:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(len(truth_set), k)

def ndcg_at_k(pred, truth, k):
    if len(truth) == 0:
        return 0.0
    pred = pred[:k]
    truth_set = set(truth)
    dcg = 0.0
    for i, p in enumerate(pred):
        if p in truth_set:
            dcg += 1.0 / np.log2(i + 2.0)
    # ideal DCG
    ideal_hits = min(len(truth_set), k)
    idcg = sum(1.0 / np.log2(i + 2.0) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

In [129]:
# заранее сохраним популярные фильмы на случай совсем пустой истории
popular_movies = (
    train_small.groupby("movie_id")["rating"]
    .count()
    .sort_values(ascending=False)
    .index.tolist()
)

def recommend_content_based(user_id, k=10, history_min_rating=4.0):
    # фильмы пользователя в train
    user_hist = train_small[train_small["user_id"] == user_id]
    if history_min_rating is not None:
        user_hist = user_hist[user_hist["rating"] >= history_min_rating]

    if user_hist.empty:
        # холодный пользователь → просто популярные
        return popular_movies[:k]

    # индексы фильмов в матрице похожести
    hist_movie_ids = [
        mid for mid in user_hist["movie_id"].values
        if mid in movie_id_to_pos
    ]

    if len(hist_movie_ids) == 0:
        return popular_movies[:k]

    hist_pos = [movie_id_to_pos[mid] for mid in hist_movie_ids]

    # суммируем похожесть ко всем фильмам
    # (n_items,) — вектор скорингов для всех фильмов
    scores = movie_sim[:, hist_pos].sum(axis=1)

    # обнулим фильмы, которые пользователь уже видел
    seen = set(train_small[train_small["user_id"] == user_id]["movie_id"].values)
    for mid in seen:
        if mid in movie_id_to_pos:
            scores[movie_id_to_pos[mid]] = -1e9

    # выбираем топ-k индексов
    top_pos = np.argsort(-scores)[:k]
    rec_movie_ids = [movie_ids_cb[i] for i in top_pos]

    return rec_movie_ids

In [130]:
def evaluate_content_based(K=10):
    recalls, maps, ndcgs = [], [], []

    # оцениваем только тех, кто есть в test
    for user_id, truth_items in user_test_items.items():
        pred = recommend_content_based(user_id, k=K)

        recalls.append(recall_at_k(pred, truth_items, K))
        maps.append(apk(pred, truth_items, K))
        ndcgs.append(ndcg_at_k(pred, truth_items, K))

    return {
        "Recall@K": float(np.mean(recalls)),
        "MAP@K": float(np.mean(maps)),
        "NDCG@K": float(np.mean(ndcgs)),
    }

cb_metrics = evaluate_content_based(K=10)
cb_metrics

{'Recall@K': 0.015124691594292231,
 'MAP@K': 0.012080665383335087,
 'NDCG@K': 0.03160682361792571}

## Объдинение kNN и content-based

In [132]:
unique_users = train_df["user_id"].unique()
user_id_to_idx = {uid: i for i, uid in enumerate(unique_users)}

In [ ]:
def recommend_hybrid(user_id, k=10, N=100, alpha=0.5):

    rec_knn = []

    if user_id in user_id_to_idx:
        user_idx = user_id_to_idx[user_id]

        # подстрахуемся: индекс должен быть в диапазоне матрицы
        n_users_knn = train_matrix.shape[0]

        if 0 <= user_idx < n_users_knn:
            try:
                rec_knn = recommend_item_knn(user_idx, k=N)
            except IndexError:
                # если вдруг внутри всё равно что-то упало – просто игнорируем kNN
                rec_knn = []
        else:
            rec_knn = []
    else:
        rec_knn = []

    rec_cb = recommend_content_based(user_id, k=N)

    # если kNN ничего не вернул – вернём чисто content-based
    if not rec_knn:
        return rec_cb[:k]


    def ranks_to_scores(items):
        scores = {}
        for rank, item in enumerate(items):
            scores[item] = len(items) - rank
        return scores

    knn_scores = ranks_to_scores(rec_knn)
    cb_scores  = ranks_to_scores(rec_cb)

    all_items = set(knn_scores.keys()) | set(cb_scores.keys())
    hybrid_scores = {}

    for item in all_items:
        s_knn = knn_scores.get(item, 0.0)
        s_cb  = cb_scores.get(item, 0.0)
        hybrid_scores[item] = (1 - alpha) * s_knn + alpha * s_cb

    sorted_items = sorted(hybrid_scores.items(), key=lambda x: -x[1])
    rec_items = [item for item, score in sorted_items[:k]]

    return rec_items

In [138]:
def evaluate_hybrid(K=10, N=100, alpha=0.5):
    recalls, maps, ndcgs = [], [], []

    for user_id, truth_items in user_test_items.items():
        pred = recommend_hybrid(user_id, k=K, N=N, alpha=alpha)

        recalls.append(recall_at_k(pred, truth_items, K))
        maps.append(apk(pred, truth_items, K))
        ndcgs.append(ndcg_at_k(pred, truth_items, K))

    return {
        "Recall@K": float(np.mean(recalls)),
        "MAP@K": float(np.mean(maps)),
        "NDCG@K": float(np.mean(ndcgs)),
    }

hybrid_metrics = evaluate_hybrid(K=10, N=100, alpha=0.5)
hybrid_metrics

{'Recall@K': 0.011296288090460285,
 'MAP@K': 0.008885128683555836,
 'NDCG@K': 0.02459484120468233}

## Сравнение моделей

Были обучены и оценены три подхода рекомендаций:
 1. Item-based kNN
 2. Content-based (на основе жанров)
 3. Hybrid (объединение kNN и content)

### Анализ результатов

Несмотря на низкие абсолютные значения метрик, результаты являются ожидаемыми с учётом свойств MovieLens 1M:
 - матрица взаимодействий крайне разреженная (≈99.9%)
 - у большинства пользователей мало пересечений
 - присутствует выраженный long-tail
 - наблюдаются эффекты холодного старта

### Наблюдения
 1. Content-based модель показывает наилучшие результаты.
Это объясняется тем, что контентный сигнал не требует пересечений между пользователями и работает независимо от sparsity.
 2. Hybrid-модель улучшает результаты item-kNN за счёт добавления контентной информации, но остаётся слабее чистого content-based, поскольку жанры являются достаточно сильным признаком.
 3. Item-based kNN показывает самые низкие значения, что типично для разреженных данных и особенно для фильмов с низкой активностью.

Вывод

Content-based подход демонстрирует наилучшее качество, hybrid — улучшает item-kNN, но уступает content-based, item-kNN — слабее остальных на разреженных пользовательских данных.

Возможные направления улучшений
 - использование TF-IDF по описаниям фильмов
 - добавление временного фактора (timestamp decay)
 - обучение модели ранжирования (LightGBM, LambdaRank)
 - использование нейронных рекоммендеров (LightFM, prod2vec, BERT4Rec и др.)
 - интеграция implicit сигналов (просмотры, клики, лайки)

## Заключение по проекту

В рамках проекта был реализован полный цикл разработки рекомендательной системы на датасете MovieLens 1M, включающий:

— анализ данных (EDA),
— построение и сравнение нескольких моделей рекомендаций,
— оценку качества по метрикам ранжирования.

EDA показал высокую разреженность данных (~99.9%) и выраженную проблему холодного старта для фильмов и частично для пользователей, а также сильный жанровый сигнал.

Из протестированных вариантов:

— Content-based модель дала наилучший результат, поскольку не требует пересечений между пользователями,
— Hybrid модель улучшила kNN, но уступила чистому content-based,
— Item-based kNN показал самый слабый результат на разреженных данных.

Несмотря на относительно низкие абсолютные значения метрик, поведение моделей является ожидаемым для рекомендательных систем на неявных взаимодействиях и разреженных данных.

Проект показал, что даже простые модели могут давать интерпретируемые результаты, а качество может быть существенно улучшено за счет более богатого контентного описания и продвинутых ранжирующих моделей.